In [ ]:
from functools import partial
import math

import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor

First we load the MNIST dataset.

In [ ]:
def load_data():
    train_ds = MNIST('data', train=True, transform=ToTensor(), download=True)
    train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
    return train_dl

Second, we create a 5-layer ReLU MLP. We set the bias to `false` to prevent exploding gradients. We also write a custom initialization function.

In [ ]:
num_layers = 5

In [ ]:
class MLP(nn.Module):
    def __init__(self, activation=nn.ReLU):
        super().__init__()
        self.layers = nn.Sequential()
        for i in range(num_layers-1):
            self.layers.add_module(f"Dense {i+1}", nn.LazyLinear(64))
            self.layers.add_module(f"Activation {i+1}", activation())
        self.layers.add_module(f"Dense {num_layers})", nn.LazyLinear(10))
        
    def forward(self, x):
        x = torch.flatten(x, start_dim=1)
        return self.layers(x)

In [ ]:
def custom_init(gain, m):
    if isinstance(m, nn.Linear):
        std = gain * math.sqrt(2 / m.weight.shape[1]) # He init
        nn.init.normal_(m.weight, std=std)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

def apply_init(model, gain):
    model(torch.zeros(1, 784)) # initialize lazy layers
    init_fn = partial(custom_init, gain)
    model.apply(init_fn)

In [ ]:
lr = 0.05
num_epochs = 1

In [ ]:
def mean(lst):
    return sum(lst) / len(lst)

def append(lst1, lst2, elem1, elem2):
    lst1.append(elem1)
    lst2.append(elem2)

In [ ]:
def stats(model, X, y):
    logits = model(X)
    loss = F.cross_entropy(logits, y)
    accuracy = (logits.argmax(dim=1) == y).float().mean()
    return loss, accuracy

def train_step(opt, model, X, y):
    loss, accuracy = stats(model, X, y)
    loss.backward()
    opt.step()
    return loss.item(), accuracy.item()

In [ ]:
def get_grads(model):
    wg, bg = [], []
    for m in model.modules():
        if isinstance(m, nn.Linear):
            wg.append(m.weight.grad.std())
            bg.append(m.bias.grad.std())
    return wg, bg

In [ ]:
def train_model(activation=nn.ReLU, gain=1.0):
    torch.manual_seed(0)
    train_dl = load_data()
    model = MLP(activation=activation)
    apply_init(model, gain=gain)
    opt = torch.optim.SGD(model.parameters(), lr)
    losses, accuracies = [], []
    weight_grads, bias_grads = [], []
    for _ in range(num_epochs):
        for i, (X, y) in enumerate(train_dl):
            loss, acc = train_step(opt, model, X, y)
            append(losses, accuracies, loss, acc)
            wg, bg = get_grads(model)
            append(weight_grads, bias_grads, wg, bg)
            opt.zero_grad()                 
    return losses, accuracies, weight_grads, bias_grads

In [ ]:
gain = 1.0

In [ ]:
losses, accuracies, wg, bg = train_model(gain=gain)

In [ ]:
losses_tanh, accuracies_tanh, wg_tanh, bg_tanh = train_model(activation=nn.Tanh, gain=gain)

In [ ]:
losses_sigmoid, accuracies_sigmoid, wg_sigmoid, bg_sigmoid = train_model(activation=nn.Sigmoid, gain=gain)

In [ ]:
wg = torch.tensor(wg).transpose(0, 1)
bg = torch.tensor(bg).transpose(0, 1)
wg_tanh = torch.tensor(wg_tanh).transpose(0, 1)
bg_tanh = torch.tensor(bg_tanh).transpose(0, 1)
wg_sigmoid = torch.tensor(wg_sigmoid).transpose(0, 1)
bg_sigmoid = torch.tensor(bg_sigmoid).transpose(0, 1)

In [ ]:
print(wg.shape)

In [ ]:
print(wg[:,0])
print(wg_tanh[:,0])
print(wg_sigmoid[:,0])
print(bg[:,0])
print(bg_tanh[:,0])
print(bg_sigmoid[:,0])

In [ ]:
fig, axs = plt.subplots(nrows=3, ncols=4, squeeze=True, figsize=(10, 8))

axs[0, 0].plot(losses)
axs[0, 1].plot(accuracies)
for i in range(wg.shape[0]):
    axs[0, 2].plot(wg[i], label=f"Layer {i}")
    axs[0, 3].plot(bg[i])
axs[0, 2].set_yscale('log')
axs[0, 3].set_yscale('log')

axs[1, 0].plot(losses_tanh)
axs[1, 1].plot(accuracies_tanh)
for i in range(wg_tanh.shape[0]):
    axs[1, 2].plot(wg_tanh[i])
    axs[1, 3].plot(bg_tanh[i])
axs[1, 2].set_yscale('log')
axs[1, 3].set_yscale('log')

axs[2, 0].plot(losses_sigmoid)
axs[2, 1].plot(accuracies_sigmoid)
for i in range(wg_sigmoid.shape[0]):
    axs[2, 2].plot(wg_sigmoid[i])
    axs[2, 3].plot(bg_sigmoid[i])
axs[2, 2].set_yscale('log')
axs[2, 3].set_yscale('log')

axs[0, 0].set_xlabel("Losses")
axs[0, 1].set_xlabel("Accuracies")
axs[1, 0].set_xlabel("Losses")
axs[1, 1].set_xlabel("Accuracies")
axs[2, 0].set_xlabel("Losses")
axs[2, 1].set_xlabel("Accuracies")
axs[0, 2].set_xlabel("Weight gradient stddev")
axs[0, 3].set_xlabel("Bias gradient stddev")
axs[1, 2].set_xlabel("Weight gradient stddev")
axs[1, 3].set_xlabel("Bias gradient stddev")
axs[2, 2].set_xlabel("Weight gradient stddev")
axs[2, 3].set_xlabel("Bias gradient stddev")
axs[0, 2].set_title("ReLU activation")
axs[0, 3].set_title("ReLU activation")
axs[1, 2].set_title("Tanh activation")
axs[1, 3].set_title("Tanh activation")
axs[2, 2].set_title("Sigmoid activation")
axs[2, 3].set_title("Sigmoid activation")

axs[0, 2].legend()
fig.tight_layout()
#plt.savefig("gradients_over_time.png")
plt.show()